# RFM analüüs UrbanStyle.ltd jaoks

**Nädal 7: Python Pandas**  
**Meeskonna töö: Data Loading, Data Cleaning, RFM Analysis, Visualization**  
**Kuupäev:** 2026-05-13

Marko Saar soovib customer-level analüüsi, et eristada VIP-kliendid, riskikliendid ja muud olulised kliendisegmendid. Selle notebooki eesmärk on laadida UrbanStyle.ltd müügi- ja kliendiandmed, puhastada need, arvutada RFM mõõdikud ning anda Markole konkreetsed soovitused.

## Roll A: Data Loading - andmete laadimine ja liitmine

Laeme müügi- ja kliendiandmed Supabase'ist ning liidame need `customer_id` põhjal üheks DataFrame'iks.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
import pandas as pd
import plotly.express as px
from supabase import create_client

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

SENSITIVE_COLUMNS = ['email', 'phone', 'first_name', 'last_name']

dotenv_candidates = [
    Path.cwd() / '.env',
    Path.cwd() / 'week7-python' / '.env',
    Path.cwd().parent / '.env',
    Path.cwd().parent / 'week7-python' / '.env',
    *[parent / '.env' for parent in Path.cwd().parents],
]
dotenv_path = next((path for path in dotenv_candidates if path.exists()), None)
load_dotenv(dotenv_path=dotenv_path)

SUPABASE_URL = os.getenv('SUPABASE_URL')
SUPABASE_KEY = os.getenv('SUPABASE_KEY') or os.getenv('SUPABASE_ANON_KEY')

if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError('Puudub SUPABASE_URL või SUPABASE_KEY .env failis')

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)


def fetch_table(table_name, page_size=1000):
    rows = []
    start = 0

    while True:
        end = start + page_size - 1
        response = supabase.table(table_name).select('*').range(start, end).execute()
        batch = response.data or []
        rows.extend(batch)

        if len(batch) < page_size:
            break

        start += page_size

    return pd.DataFrame(rows)


def display_public_sample(frame, columns, rows=5):
    safe_columns = [column for column in columns if column in frame.columns]
    display(frame[safe_columns].head(rows))


df_sales = fetch_table('sales')
df_customers = fetch_table('customers')

print('\nSales shape:', df_sales.shape)
display_public_sample(df_sales, ['sale_id', 'customer_id', 'sale_date', 'product_id', 'quantity', 'total_price', 'store_location'])
print('\nCustomers shape:', df_customers.shape)
display_public_sample(df_customers, ['customer_id', 'city', 'loyalty_tier', 'registration_date'])

df = pd.merge(df_sales, df_customers, on='customer_id', how='left')

print('\nLiidetud DataFrame shape:', df.shape)
print('\nLiidetud DataFrame dtypes, excluding sensitive columns:')
public_dtype_columns = [column for column in df.columns if column not in SENSITIVE_COLUMNS]
print(df[public_dtype_columns].dtypes)
display_public_sample(df, ['sale_id', 'customer_id', 'sale_date', 'product_id', 'quantity', 'total_price', 'store_location', 'city', 'loyalty_tier'])

required_columns = ['customer_id', 'sale_date', 'total_price']
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f'Puuduvad kohustuslikud veerud: {missing_columns}')

print('\nKontroll läbitud: customer_id, sale_date ja total_price on olemas.')

## Roll B: Data Cleaning - puhastamine ja valideerimine

Eemaldame duplikaadid, käsitleme kriitilised NULL väärtused, parsime kuupäevad ning jätame alles ainult positiivse `total_price` väärtusega read.

In [ ]:
print('Esialgne shape:', df.shape)

initial_rows = len(df)
duplicates_before = df.duplicated().sum()
print('Duplikaadid enne eemaldamist:', duplicates_before)

df_clean = df.drop_duplicates().copy()
duplicates_after = df_clean.duplicated().sum()
print('Duplikaadid pärast eemaldamist:', duplicates_after)

print('\nNULL-id enne puhastamist:')
nulls_before = df_clean.isnull().sum()
public_nulls_before = nulls_before.drop(labels=SENSITIVE_COLUMNS, errors='ignore')
print(public_nulls_before[public_nulls_before > 0].sort_values(ascending=False))

critical_columns = ['customer_id', 'sale_date', 'total_price']
critical_nulls_before = df_clean[critical_columns].isnull().sum()
df_clean = df_clean.dropna(subset=critical_columns).copy()

df_clean['sale_date'] = pd.to_datetime(df_clean['sale_date'], errors='coerce')
invalid_dates = df_clean['sale_date'].isna().sum()
df_clean = df_clean.dropna(subset=['sale_date']).copy()

df_clean['total_price'] = pd.to_numeric(df_clean['total_price'], errors='coerce')
invalid_prices = df_clean['total_price'].isna().sum()
df_clean = df_clean.dropna(subset=['total_price']).copy()

non_positive_prices = (df_clean['total_price'] <= 0).sum()
df_clean = df_clean[df_clean['total_price'] > 0].copy()

final_rows = len(df_clean)
removed_rows = initial_rows - final_rows

print('\nPuhastusraport')
print('-' * 40)
print(f'Algne ridade arv: {initial_rows}')
print(f'Eemaldatud duplikaate: {duplicates_before}')
print('Kriitilised NULL-id enne eemaldamist:')
print(critical_nulls_before)
print(f'Vigaseid kuupäevi pärast parsimist: {invalid_dates}')
print(f'Mittearvulisi total_price väärtusi: {invalid_prices}')
print(f'Mittepositiivseid total_price väärtusi: {non_positive_prices}')
print(f'Eemaldatud ridu kokku: {removed_rows}')
print(f'Lõplik shape: {df_clean.shape}')
print(f'Unikaalseid kliente: {df_clean["customer_id"].nunique()}')
print(f'Kuupäevavahemik: {df_clean["sale_date"].min().date()} kuni {df_clean["sale_date"].max().date()}')

print('\nKontroll: kriitiliste veergude NULL-id pärast puhastamist')
print(df_clean[critical_columns].isnull().sum())
print('\nsale_date tüüp:', df_clean['sale_date'].dtype)
print('Min total_price:', df_clean['total_price'].min())

df = df_clean.reset_index(drop=True)
display_public_sample(df, ['sale_id', 'customer_id', 'sale_date', 'product_id', 'quantity', 'total_price', 'store_location', 'city', 'loyalty_tier'])

## Roll C: RFM Analysis - Recency, Frequency, Monetary ja segmendid

Arvutame iga kliendi kohta RFM väärtused, määrame kvintiilide alusel skoorid 1-5 ning loome kliendisegmendid.

In [ ]:
today = pd.to_datetime('2025-02-28')

rfm = (
    df.groupby('customer_id')
    .agg(
        last_purchase_date=('sale_date', 'max'),
        frequency=('sale_id', 'count'),
        monetary_value=('total_price', 'sum'),
        city=('city', 'first'),
        loyalty_tier=('loyalty_tier', 'first'),
    )
    .reset_index()
)

rfm['recency_days'] = (today - rfm['last_purchase_date']).dt.days

rfm['R_score'] = pd.qcut(
    rfm['recency_days'].rank(method='first'),
    5,
    labels=[5, 4, 3, 2, 1],
).astype(int)
rfm['F_score'] = pd.qcut(
    rfm['frequency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5],
).astype(int)
rfm['M_score'] = pd.qcut(
    rfm['monetary_value'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5],
).astype(int)

rfm['RFM_Score'] = rfm[['R_score', 'F_score', 'M_score']].sum(axis=1)


def assign_segment(score):
    if score >= 13:
        return 'VIP Champions'
    if score >= 10:
        return 'Loyal'
    if score >= 7:
        return 'Potential'
    if score >= 4:
        return 'At Risk'
    return 'Lost'


rfm['Segment'] = rfm['RFM_Score'].apply(assign_segment)

segment_summary = (
    rfm.groupby('Segment')
    .agg(
        customers=('customer_id', 'count'),
        avg_recency_days=('recency_days', 'mean'),
        avg_frequency=('frequency', 'mean'),
        total_revenue=('monetary_value', 'sum'),
        avg_monetary_value=('monetary_value', 'mean'),
    )
    .reset_index()
)
segment_summary['customer_share_pct'] = segment_summary['customers'] / segment_summary['customers'].sum() * 100
segment_summary['revenue_share_pct'] = segment_summary['total_revenue'] / segment_summary['total_revenue'].sum() * 100
segment_summary = segment_summary.sort_values('total_revenue', ascending=False)

print('RFM tabel shape:', rfm.shape)
print('\nSkooride vahemikud:')
print(rfm[['R_score', 'F_score', 'M_score', 'RFM_Score']].agg(['min', 'max']))

print('\nSegmentide kokkuvõte:')
display(segment_summary.round(2))

display(rfm.sort_values('RFM_Score', ascending=False).head(10)[[
    'customer_id', 'Segment', 'recency_days', 'frequency', 'monetary_value',
    'R_score', 'F_score', 'M_score', 'RFM_Score', 'city', 'loyalty_tier'
]])

## Roll D: Visualization - diagrammid ja leiud

Loome kolm Plotly diagrammi: segmentide jaotus, RFM hajuvusdiagramm ning top 10 VIP klienti kogukulutuse järgi.

In [ ]:
segment_order = ['VIP Champions', 'Loyal', 'Potential', 'At Risk', 'Lost']

fig_segments = px.bar(
    segment_summary.sort_values('customers', ascending=False),
    x='Segment',
    y='customers',
    text='customers',
    title='UrbanStyle kliendisegmentide jaotus',
    labels={'Segment': 'Segment', 'customers': 'Klientide arv'},
    color='Segment',
    category_orders={'Segment': segment_order},
)
fig_segments.update_layout(showlegend=False)
fig_segments.update_traces(textposition='outside')
fig_segments.show()

fig_scatter = px.scatter(
    rfm,
    x='recency_days',
    y='monetary_value',
    color='Segment',
    size='frequency',
    hover_data=['customer_id', 'frequency', 'RFM_Score'],
    title='UrbanStyle kliendisegmendid RFM analüüsi põhjal',
    labels={
        'recency_days': 'Päevi viimasest ostust',
        'monetary_value': 'Kogukulutus (EUR)',
        'frequency': 'Ostude arv',
        'Segment': 'Segment',
    },
    category_orders={'Segment': segment_order},
)
fig_scatter.show()

top_vip = rfm[rfm['Segment'] == 'VIP Champions'].nlargest(10, 'monetary_value').copy()
top_vip['customer_label'] = 'Customer ' + top_vip['customer_id'].astype(str)

fig_vip = px.bar(
    top_vip.sort_values('monetary_value'),
    x='monetary_value',
    y='customer_label',
    orientation='h',
    text='monetary_value',
    title='Top 10 VIP klienti kogukulutuse järgi',
    labels={'monetary_value': 'Kogukulutus (EUR)', 'customer_label': 'Klient'},
)
fig_vip.update_traces(texttemplate='%{text:.0f} EUR', textposition='outside')
fig_vip.show()

<style>
.jp-RenderedHTMLCommon .github-only,
.rendered_html .github-only {
  display: none;
}
</style>

<div class="github-only">
<h3>Roll D visualisatsioonid GitHubi jaoks</h3>
<p>GitHub note: open this notebook in Jupyter or VS Code to view the interactive Plotly RFM visualizations.</p>


</div>


## Äritõlgendus Markole

In [ ]:
vip_customers = int((rfm['Segment'] == 'VIP Champions').sum())
at_risk_customers = int((rfm['Segment'] == 'At Risk').sum())
lost_customers = int((rfm['Segment'] == 'Lost').sum())
total_customers = int(rfm['customer_id'].nunique())
total_revenue = rfm['monetary_value'].sum()
vip_revenue = rfm.loc[rfm['Segment'] == 'VIP Champions', 'monetary_value'].sum()
vip_revenue_share = vip_revenue / total_revenue * 100
at_risk_revenue = rfm.loc[rfm['Segment'] == 'At Risk', 'monetary_value'].sum()
at_risk_revenue_share = at_risk_revenue / total_revenue * 100

interpretation = f'''
UrbanStyle andmestikus on {total_customers} analüüsitavat klienti, kellest {vip_customers} kuuluvad VIP Champions segmenti.
VIP kliendid annavad {vip_revenue_share:.1f}% kogukäibest, seega on nad Marko jaoks kõige väärtuslikum hoidmise ja erikohtlemise sihtrühm.
At Risk segmendis on {at_risk_customers} klienti ja Lost segmendis {lost_customers} klienti; nende puhul on peamine risk ostmise vähenemine või lõppemine.
At Risk segment annab veel {at_risk_revenue_share:.1f}% käibest, mistõttu tasub nendega kiiresti teha personaliseeritud win-back kampaania.
'''

recommendations = '''
Soovitused:
1. VIP Champions: käivita early access programm, personaalsed pakkumised ja VIP sooduskoodid.
2. At Risk: saada "Me igatseme sind" tüüpi win-back e-mail koos piiratud ajaga personaalse pakkumisega.
3. Potential ja Loyal: kasuta lojaalsusprogrammi, cross-sell pakkumisi ja kordusostu soodustusi, et kasvatada nad VIP segmendiks.
'''

print(interpretation.strip())
print()
print(recommendations.strip())

## Edasijõudnute lisa: segmentide eksport

Salvestame RFM segmentide tabeli CSV-failina, et Marko saaks selle edasi anda turundusmeeskonnale.

In [ ]:
export_columns = [
    'customer_id', 'city', 'loyalty_tier', 'last_purchase_date',
    'recency_days', 'frequency', 'monetary_value', 'R_score', 'F_score', 'M_score',
    'RFM_Score', 'Segment'
]
export_path = Path('week7-python/team/rfm_segments.csv')
if not export_path.parent.exists():
    export_path = Path('rfm_segments.csv')

rfm[export_columns].to_csv(export_path, index=False, encoding='utf-8')
print(f'RFM segmendid salvestatud: {export_path}')

## Kokkuvõte: soovitused Markole ja meeskonna refleksioon

**Mis oli suurim üllatus?**  
VIP Champions segment ei ole kõige suurem kliendigrupp, kuid selle käibeosakaal on väga suur. See näitab, et väike osa klientidest võib kanda ebaproportsionaalselt suurt osa müügitulust.

**Milline on meie soovitus Markole?**  
Kõige kiiremini tuleks tegutseda kahe segmendiga: VIP Champions ja At Risk. VIP klientidele tuleb pakkuda eksklusiivseid eeliseid, et neid hoida; At Risk klientidele tuleb teha personaliseeritud win-back kampaania enne, kui nad liiguvad Lost segmenti.

**Milliseid andmeid meil puudus?**  
Analüüsi muudaks paremaks info tagastuste, kampaaniate, kliendi veebikäitumise, e-mailide avamise ja NPS rahuloluskoori kohta. Need aitaksid paremini selgitada, miks osa kliente lõpetab ostmise.